In [0]:

import datetime as dt
import subprocess
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import Window


In [0]:
# ============================================================
# FUNCTION 1: get_initial_data()
# Fetch 720 days of hourly OHLCV data for any ticker
# ============================================================

def get_initial_data(ticker: str) -> pd.DataFrame:
    """
    Fetches 720 days of hourly OHLCV data from Yahoo Finance.

    Args:
        ticker (str): Any valid Yahoo Finance ticker 
                      e.g. 'BTC-USD', 'ETH-USD', 'AAPL', 'TSLA'
    Returns:
        pd.DataFrame: Raw OHLCV data — nothing modified
    """
    try:
        import yfinance as yf
    except ImportError:
        subprocess.check_call(["pip", "install", "-q", "yfinance"])
        import yfinance as yf

    end_date   = dt.datetime.now(dt.timezone.utc)
    start_date = end_date - dt.timedelta(days=720)

    print(f"📥 Fetching {ticker} | {start_date.strftime('%Y-%m-%d')} → {end_date.strftime('%Y-%m-%d')} | interval=1h")

    raw_df = yf.download(
        tickers=ticker,
        start=start_date.strftime('%Y-%m-%d'),
        end=end_date.strftime('%Y-%m-%d'),
        interval="1h",
        auto_adjust=False,
        progress=False
    )

    if raw_df is None or len(raw_df) == 0:
        raise ValueError(f"yfinance returned no data for {ticker}.")

    print(f"✅ Downloaded {len(raw_df)} rows for {ticker}.")
    return raw_df

In [0]:
# ============================================================
# FUNCTION 2: store_bronze()
# Store raw_df into catalog_{ticker}.bronze.raw
# ============================================================

def store_bronze(raw_df: pd.DataFrame, ticker: str, spark) -> None:
    """
    Stores raw yfinance data into Bronze layer as-is.
    Only flattens MultiIndex columns (Spark requirement)
    and renames 'Adj Close' → 'Adj_Close' (Delta requirement).
    No other modifications.

    Catalog : {ticker_clean}
    Schema  : bronze
    Table   : raw

    Args:
        raw_df (pd.DataFrame) : Raw DataFrame from get_initial_data()
        ticker (str)          : Ticker symbol e.g. 'BTC-USD'
        spark  (SparkSession) : Active Spark session
    """
    # Build catalog name from ticker e.g. 'BTC-USD' → 'btc_usd'
    catalog = ticker.lower().replace("-", "_").replace(".", "_")
    table   = f"{catalog}.bronze.raw"

    # Create catalog and schema if not exists
    spark.sql(f"CREATE CATALOG  IF NOT EXISTS {catalog}")
    spark.sql(f"CREATE SCHEMA   IF NOT EXISTS {catalog}.bronze")

    df = raw_df.copy()

    # Flatten MultiIndex (Spark can't handle tuple column names)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    # Fix invalid Delta column name
    df = df.rename(columns={"Adj Close": "Adj_Close"})

    # Add ingestion timestamp for auditing
    df["ingested_at"] = dt.datetime.now(dt.timezone.utc)

    # Reset index to bring Datetime in as a column
    df = df.reset_index()

    sdf = spark.createDataFrame(df)

    try:
        latest_ts = spark.table(table).agg(F.max("Datetime")).first()[0]
        new_rows  = sdf.filter(F.col("Datetime") > latest_ts)
        new_count = new_rows.count()
        if new_count == 0:
            print(f"ℹ️  Bronze: no new rows for {ticker}.")
            return
        new_rows.write.mode("append").saveAsTable(table)
        print(f"🥉 Bronze [{table}]: appended {new_count} rows.")
    except Exception:
        sdf.write.mode("overwrite").saveAsTable(table)
        print(f"🥉 Bronze [{table}]: first load — wrote {sdf.count()} rows.")

In [0]:
def store_silver(ticker: str, spark) -> None:
    catalog      = ticker.lower().replace("-", "_").replace(".", "_")
    bronze_table = f"{catalog}.bronze.raw"
    silver_table = f"{catalog}.silver.cleaned"

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")

    # ----------------------------------------------------------
    # CLEAN
    # ----------------------------------------------------------
    df = spark.table(bronze_table)

    for col in df.columns:
        df = df.withColumnRenamed(col, col.lower())

    df = (
        df
        .drop("ingested_at")
        .withColumn("datetime", F.to_timestamp(F.col("datetime")))
        .withColumn("open",     F.col("open").cast("double"))
        .withColumn("high",     F.col("high").cast("double"))
        .withColumn("low",      F.col("low").cast("double"))
        .withColumn("close",    F.col("close").cast("double"))
        .withColumn("volume",   F.col("volume").cast("double"))
        .filter(F.col("datetime").isNotNull())
        .filter(F.col("close").isNotNull())
        .filter(F.col("close") > 0)
    )

    # Deduplicate by keeping row with highest volume per timestamp
    w_dedup = Window.partitionBy("datetime").orderBy(F.desc("volume"))
    df = (
        df
        .withColumn("rank", F.row_number().over(w_dedup))
        .filter(F.col("rank") == 1)
        .drop("rank")
        .orderBy("datetime")
        .select("datetime", "open", "high", "low", "close", "volume")
    )

    # ----------------------------------------------------------
    # EDA FEATURES
    # ----------------------------------------------------------
    w_order  = Window.orderBy("datetime")
    w_ma7    = w_order.rowsBetween(-6,  0)
    w_ma21   = w_order.rowsBetween(-20, 0)
    w_rsi14  = w_order.rowsBetween(-13, 0)
    w_bb20   = w_order.rowsBetween(-19, 0)
    w_vol24  = w_order.rowsBetween(-23, 0)
    w_std24  = w_order.rowsBetween(-23, 0)

    # Base return
    df = (
        df
        .withColumn("prev_close", F.lag("close", 1).over(w_order))
        .withColumn("return_1h",  (F.col("close") - F.col("prev_close")) / F.col("prev_close"))
    )

    # Lag returns
    df = (
        df
        .withColumn("lag_return_2h",  F.lag("return_1h",  2).over(w_order))
        .withColumn("lag_return_3h",  F.lag("return_1h",  3).over(w_order))
        .withColumn("lag_return_6h",  F.lag("return_1h",  6).over(w_order))
        .withColumn("lag_return_24h", F.lag("return_1h", 24).over(w_order))
    )

    # Moving averages
    df = (
        df
        .withColumn("ma7",  F.avg("close").over(w_ma7))
        .withColumn("ma21", F.avg("close").over(w_ma21))
    )

    # RSI14
    df = (
        df
        .withColumn("delta",       F.col("close") - F.col("prev_close"))
        .withColumn("gain",        F.when(F.col("delta") > 0,  F.col("delta")).otherwise(F.lit(0.0)))
        .withColumn("loss",        F.when(F.col("delta") < 0, -F.col("delta")).otherwise(F.lit(0.0)))
        .withColumn("avg_gain_14", F.avg("gain").over(w_rsi14))
        .withColumn("avg_loss_14", F.avg("loss").over(w_rsi14))
        .withColumn(
            "rsi14",
            F.when(F.col("avg_loss_14") == 0, F.lit(100.0))
             .otherwise(100.0 - (100.0 / (1.0 + (F.col("avg_gain_14") / F.col("avg_loss_14")))))
        )
    )

    # Bollinger Bands
    df = (
        df
        .withColumn("bb_ma20",  F.avg("close").over(w_bb20))
        .withColumn("bb_std20", F.stddev("close").over(w_bb20))
        .withColumn("bb_upper", F.col("bb_ma20") + (2 * F.col("bb_std20")))
        .withColumn("bb_lower", F.col("bb_ma20") - (2 * F.col("bb_std20")))
        .withColumn("bb_width", F.col("bb_upper") - F.col("bb_lower"))
    )

    # Time features
    df = (
        df
        .withColumn("hour_of_day", F.hour("datetime").cast("double"))
        .withColumn("day_of_week", F.dayofweek("datetime").cast("double"))
    )

    # Volatility — 24h rolling std of returns
    df = df.withColumn("volatility_24h", F.stddev("return_1h").over(w_std24))

    # Momentum — 6h price change
    df = df.withColumn(
        "momentum_6h",
        F.when(
            F.lag("close", 6).over(w_order) == 0, F.lit(0.0)
        ).otherwise(
            (F.col("close") - F.lag("close", 6).over(w_order)) / F.lag("close", 6).over(w_order)
        )
    )

    # Volume features
    df = (
        df
        .withColumn("volume_ma24",  F.avg("volume").over(w_vol24))
        .withColumn("volume_ratio", 
            F.when(F.col("volume_ma24") == 0, F.lit(1.0))  # default to 1.0 if avg is 0
             .otherwise(F.col("volume") / F.col("volume_ma24"))
        )
    )
    # Final select
    silver = df.select(
        "datetime", "open", "high", "low", "close", "volume",
        "return_1h",
        "lag_return_2h", "lag_return_3h", "lag_return_6h", "lag_return_24h",
        "ma7", "ma21",
        "rsi14",
        "bb_upper", "bb_lower", "bb_width",
        "hour_of_day", "day_of_week",
        "volatility_24h", "momentum_6h",
        "volume_ma24", "volume_ratio"
    )

    silver.write.mode("overwrite").saveAsTable(silver_table)
    print(f"🥈 Silver [{silver_table}]: wrote {silver.count()} rows.")

    # ----------------------------------------------------------
    # EDA Summary
    # ----------------------------------------------------------
    print(f"\n📊 EDA Summary [{ticker}]:")
    spark.table(silver_table).select(
        F.round(F.avg("close"),         2).alias("avg_close"),
        F.round(F.min("close"),         2).alias("min_close"),
        F.round(F.max("close"),         2).alias("max_close"),
        F.round(F.avg("return_1h"),     6).alias("avg_return_1h"),
        F.round(F.avg("rsi14"),         2).alias("avg_rsi14"),
        F.round(F.avg("bb_width"),      2).alias("avg_bb_width"),
        F.round(F.avg("volatility_24h"),6).alias("avg_volatility_24h"),
        F.round(F.avg("momentum_6h"),   6).alias("avg_momentum_6h"),
        F.round(F.avg("volume_ratio"),  4).alias("avg_volume_ratio"),
    ).display()

In [0]:
def run_hourly_job(ticker: str, spark) -> None:
    try:
        import yfinance as yf
    except ImportError:
        subprocess.check_call(["pip", "install", "-q", "yfinance"])
        import yfinance as yf

    catalog      = ticker.lower().replace("-", "_").replace(".", "_")
    bronze_table = f"{catalog}.bronze.raw"

    print("=" * 55)
    print(f"Hourly Job | {ticker} | {dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print("=" * 55)

    end_date = dt.datetime.now(dt.timezone.utc)

    # ----------------------------------------------------------
    # STEP 1 — Find latest timestamp in Bronze
    # ----------------------------------------------------------
    try:
        latest_ts = spark.table(bronze_table).agg(F.max("Datetime")).first()[0]

        if latest_ts is None:
            raise Exception("Empty table")

        # Normalize to UTC-aware datetime regardless of type
        if isinstance(latest_ts, dt.datetime):
            if latest_ts.tzinfo is None:
                latest_ts = latest_ts.replace(tzinfo=dt.timezone.utc)
        else:
            latest_ts = latest_ts.to_pydatetime().replace(tzinfo=dt.timezone.utc)

        start_date = latest_ts
        print(f"📅 Last Bronze timestamp : {latest_ts}")
        print(f"📅 Fetching from         : {start_date.strftime('%Y-%m-%d')}")

    except Exception as e:
        print(f"⚠️  Exception: {e}")
        start_date = end_date - dt.timedelta(days=720)
        latest_ts  = None
        print("📅 Bronze empty — fetching full 720 days.")

    # ----------------------------------------------------------
    # STEP 2 — Fetch only new rows
    # ----------------------------------------------------------
    raw_df = yf.download(
        tickers=ticker,
        start=start_date.strftime('%Y-%m-%d'),
        end=end_date.strftime('%Y-%m-%d'),
        interval="1h",
        auto_adjust=False,
        progress=False
    )

    if raw_df is None or len(raw_df) == 0:
        print(f"ℹ️  No new data for {ticker} — job skipped.")
        return

    print(f"✅ Fetched {len(raw_df)} rows.")

    # ----------------------------------------------------------
    # STEP 3 — Append only strictly new rows
    # ----------------------------------------------------------
    df = raw_df.copy()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]

    df = df.rename(columns={"Adj Close": "Adj_Close"})
    df["ingested_at"] = dt.datetime.now(dt.timezone.utc)
    df = df.reset_index()

    sdf = spark.createDataFrame(df)

    if latest_ts is not None:
        # Cast to UTC in Spark for correct comparison
        new_rows = sdf.filter(
            F.col("Datetime") > F.lit(latest_ts).cast("timestamp")
        )
    else:
        new_rows = sdf

    count = new_rows.count()
    print(f"🔍 New rows found: {count}")

    if count == 0:
        print("ℹ️  Bronze: already up to date — job skipped.")
        return

    new_rows.write.mode("append").saveAsTable(bronze_table)
    print(f"🥉 Bronze: appended {count} new rows.")

    # ----------------------------------------------------------
    # STEP 4 — Refresh Silver
    # ----------------------------------------------------------
    print("\n🧹 Refreshing Silver...")
    store_silver(ticker, spark)

    print(f"\n🏁 Job complete for {ticker}!")

In [0]:
# raw_df = get_initial_data("BTC-USD")
# store_bronze(raw_df, "BTC-USD", spark)
# store_silver("BTC-USD", spark)

In [0]:
run_hourly_job("BTC-USD",spark)  